# Training and test sets - NHL Play-By-Play

This is the demo to obtain the training and test sets inside your NHLData folder to be used for the ML experiments

In [1]:
from ift6758.data import SeasonData
from ift6758.data.data_cleaning import clean_play_by_play_data, clean_season_data, feature_engineering_1, feature_engineering_2
import pandas as pd
import os

In [3]:
# Obtain the data from the regular seasons of 2016/17 to 2020/21
dfs = {}

for year in [2016, 2017, 2018, 2019, 2020]:
    season = SeasonData(year)
    season.get_data_from_api()

    all_events = []

    for game_id, game_data in season.reg_season_data.items():
        try:
            game_df = clean_play_by_play_data(game_data)
            game_df['season'] = season.year
            all_events.append(game_df)
        except Exception as e:
            print(f"Error processing regular season game {game_id}: {e}")

    if all_events:
        combined_df = pd.concat(all_events, ignore_index=True)

        combined_df = combined_df.sort_values(['game_id', 'period', 'period_time'])

    else:
        combined_df = pd.DataFrame(columns=[
            'game_id', 'period', 'period_time', 'team', 'event_type',
            'x_coord', 'y_coord', 'standardized_x_coord', 'standardized_y_coord',
            'shooter', 'goalie', 'shot_type',
            'empty_net', 'strength', 'season', 'prev_play_event_type', 'prev_play_period_time',
            'prev_play_x_coord', 'prev_play_y_coord', 'prev_play_standardized_x_coord',
            'prev_play_standardized_y_cood'
        ])
    
    dfs[year] = combined_df


In [4]:
# The seasons 2016/17 to 2019/20 will serve as our training set
train_df = pd.concat([dfs[2016], dfs[2017], dfs[2018], dfs[2019]], axis=0)

# The training and test sets will be saved in ift6758/data/NHLData
dir = os.getcwd()
root = os.path.dirname(dir)

datapath = os.path.join(root, "ift6758", "data", "NHLData")

train_df.to_csv(os.path.join(datapath, "training_set.csv"), index=False)
dfs[2020].to_csv(os.path.join(datapath, "test_set.csv"), index=False)

In [5]:
# Get the first "feature-engineered" train and test sets, if desired

feature_engineered_train_df_1 = feature_engineering_1(train_df)
feature_engineered_test_df_1 = feature_engineering_1(dfs[2020])

feature_engineered_train_df_1.to_csv(os.path.join(datapath, "feature_engineered_training_set_1.csv"), index=False)
feature_engineered_test_df_1.to_csv(os.path.join(datapath, "feature_engineered_test_set_1.csv"), index=False)

In [6]:
# Get the second "feature-engineered" train and test sets, if desired
feature_engineered_train_df_2 = feature_engineering_2(train_df)
feature_engineered_test_df_2 = feature_engineering_2(dfs[2020])

feature_engineered_train_df_2.to_csv(os.path.join(datapath, "feature_engineered_training_set_2.csv"), index=False)
feature_engineered_test_df_2.to_csv(os.path.join(datapath, "feature_engineered_test_set_2.csv"), index=False)